# LSTM Music Generation
End-to-end: tokenization → training → MIDI generation using REMI + MusicLSTM on MAESTRO.

In [ ]:
%pip install -q torch miditok symusic tqdm

## Imports

In [ ]:
import os
import glob
import math
import random
import types

# Windows fix: prevent OpenMP duplicate-library crash
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from miditok import REMI, TokenizerConfig

## Config

In [ ]:
cfg = types.SimpleNamespace(
    # hardware
    device="cuda" if torch.cuda.is_available() else "cpu",

    # data
    data_dir="../maestro_midi",
    file_limit=None,
    seq_len=512,
    val_split=0.1,

    # model
    d_model=512,
    n_layers=2,
    dropout=0.3,

    # training
    batch_size=16,
    lr=0.001,
    epochs=100,

    # output
    save_dir="./outputs",
    save_path="./outputs/lstm_best.pt",
)

print(f"Device: {cfg.device}")
print(f"Data: {cfg.data_dir} (file_limit={cfg.file_limit})")
print(f"Model: d_model={cfg.d_model} n_layers={cfg.n_layers} dropout={cfg.dropout}")
print(f"Train: epochs={cfg.epochs} lr={cfg.lr} batch={cfg.batch_size}")

## Tokenizer
REMI tokenizer with a compact ~240-token vocabulary. Chords and time signatures excluded to keep the task tractable for an LSTM.

In [ ]:
def build_tokenizer() -> REMI:
    config = TokenizerConfig(
        num_velocities=16,
        use_chords=False,
        use_rests=True,
        use_tempos=True,
        use_time_signatures=False,
        use_programs=False,
    )
    return REMI(config)


tokenizer = build_tokenizer()
vocab_size = len(tokenizer)
print(f"Vocab size: {vocab_size}")

## Data Loading

In [ ]:
def load_and_tokenize(
    tokenizer: REMI,
    data_dir: str,
    file_limit: int | None = None,
    seed: int = 42,
) -> list[list[int]]:
    paths = (
        glob.glob(os.path.join(data_dir, "**/*.mid"), recursive=True) +
        glob.glob(os.path.join(data_dir, "**/*.midi"), recursive=True)
    )
    if not paths:
        raise FileNotFoundError(f"No MIDI files found in {data_dir}")

    random.seed(seed)
    random.shuffle(paths)
    if file_limit is not None:
        paths = paths[:file_limit]

    sequences = []
    for path in tqdm(paths, desc="Tokenizing"):
        try:
            tokens = tokenizer(path)
            sequences.append(tokens[0].ids)
        except Exception as e:
            print(f"  Skipping {os.path.basename(path)}: {e}")

    print(f"Loaded {len(sequences)} sequences from {len(paths)} files")
    return sequences


def train_val_split(
    sequences: list[list[int]],
    val_ratio: float = 0.1,
) -> tuple[list[list[int]], list[list[int]]]:
    n_val = max(1, int(len(sequences) * val_ratio))
    return sequences[n_val:], sequences[:n_val]


class MusicDataset(Dataset):
    def __init__(self, sequences: list[list[int]], seq_len: int):
        self.samples: list[torch.Tensor] = []
        stride = seq_len // 2
        for seq in sequences:
            if len(seq) < seq_len + 1:
                continue
            for i in range(0, len(seq) - seq_len, stride):
                chunk = seq[i : i + seq_len + 1]
                self.samples.append(torch.tensor(chunk, dtype=torch.long))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        chunk = self.samples[idx]
        return chunk[:-1], chunk[1:]

In [ ]:
sequences = load_and_tokenize(tokenizer, cfg.data_dir, cfg.file_limit)
train_seqs, val_seqs = train_val_split(sequences, cfg.val_split)

train_set = MusicDataset(train_seqs, cfg.seq_len)
val_set = MusicDataset(val_seqs, cfg.seq_len)

train_loader = DataLoader(train_set, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_set):,}")
print(f"Val samples: {len(val_set):,}")

## Model
LSTM language model with weight-tying between the input embedding and output projection.

In [ ]:
class MusicLSTM(nn.Module):

    def __init__(self, vocab_size: int, d_model: int = 512, n_layers: int = 2, dropout: float = 0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.embed_drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=d_model,
            num_layers=n_layers,
            dropout=dropout if n_layers > 1 else 0.0,
            batch_first=True,
        )
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(d_model, vocab_size, bias=False)
        self.fc.weight = self.embedding.weight

    def forward(self, src: torch.Tensor):
        x = self.embed_drop(self.embedding(src))
        out, _ = self.lstm(x)
        out = self.drop(out)
        return self.fc(out)


model = MusicLSTM(
    vocab_size=vocab_size,
    d_model=cfg.d_model,
    n_layers=cfg.n_layers,
    dropout=cfg.dropout,
).to(cfg.device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")
print(model)

## Training

In [ ]:
os.makedirs(cfg.save_dir, exist_ok=True)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

best_val_loss = float("inf")

for epoch in range(cfg.epochs):

    model.train()
    train_loss = 0.0

    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs} train"):
        x, y = x.to(cfg.device), y.to(cfg.device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits.view(-1, vocab_size), y.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()

    avg_train = train_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in tqdm(val_loader, desc=f"Epoch {epoch+1}/{cfg.epochs} val"):
            x, y = x.to(cfg.device), y.to(cfg.device)
            logits = model(x)
            val_loss += criterion(logits.view(-1, vocab_size), y.view(-1)).item()

    avg_val = val_loss / len(val_loader)
    val_ppl = math.exp(min(avg_val, 20))

    print(f"Epoch {epoch+1:3d} | Train {avg_train:.4f} | Val {avg_val:.4f} | PPL {val_ppl:.1f}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "val_loss": avg_val,
                "val_ppl": val_ppl,
                "config": {
                    "vocab_size": vocab_size,
                    "d_model": cfg.d_model,
                    "n_layers": cfg.n_layers,
                    "dropout": cfg.dropout,
                },
            },
            cfg.save_path,
        )
        print(f"  Saved best model -> {cfg.save_path}")

## Generation
Grammar-constrained autoregressive sampling. Tokens are filtered at each step to enforce valid REMI ordering: `Bar -> Position -> Pitch -> Velocity -> Duration`.

In [ ]:
CHECKPOINT_PATH = cfg.save_path
OUTPUT_PATH = "./outputs/generated.mid"
MAX_NEW_TOKENS = 2000
TEMPERATURE = 0.8
TOP_K = 30

In [ ]:
def get_token_groups(tok) -> dict[str, list[int]]:
    groups = {"pitch": [], "velocity": [], "duration": [], "structural": []}
    for name, tid in tok.vocab.items():
        if name.startswith("Pitch_"):
            groups["pitch"].append(tid)
        elif name.startswith("Velocity_"):
            groups["velocity"].append(tid)
        elif name.startswith("Duration_"):
            groups["duration"].append(tid)
        elif name.startswith(("Bar_", "Position_", "Tempo_", "Rest_")):
            groups["structural"].append(tid)
    return groups


def next_phase(token_name: str) -> str:
    if token_name.startswith("Pitch_"):
        return "velocity"
    if token_name.startswith("Velocity_"):
        return "duration"
    return "pitch"


def sample_token(logits: torch.Tensor, allowed: list[int], temperature: float, top_k: int) -> int:
    if allowed:
        mask = torch.full_like(logits, float("-inf"))
        mask[torch.tensor(allowed, device=logits.device)] = logits[allowed]
        logits = mask
    logits = logits / temperature
    top_vals, top_idx = torch.topk(logits, min(top_k, logits.size(-1)))
    probs = torch.softmax(top_vals, dim=-1)
    return top_idx[torch.multinomial(probs, 1)].item()

In [ ]:
@torch.no_grad()
def generate(checkpoint_path, output_path, max_new_tokens=2000, temperature=0.8, top_k=30):
    tok = build_tokenizer()
    vocab = tok.vocab
    rev_vocab = {v: k for k, v in vocab.items()}
    groups = get_token_groups(tok)

    ckpt = torch.load(checkpoint_path, map_location=cfg.device, weights_only=True)
    saved_cfg = ckpt.get("config", {})
    gen_model = MusicLSTM(
        vocab_size=saved_cfg.get("vocab_size", len(tok)),
        d_model=saved_cfg.get("d_model", cfg.d_model),
        n_layers=saved_cfg.get("n_layers", cfg.n_layers),
        dropout=0.0,
    ).to(cfg.device)
    gen_model.load_state_dict(ckpt["model_state_dict"])
    gen_model.eval()
    print(f"Loaded epoch {ckpt.get('epoch', '?')}  val PPL {ckpt.get('val_ppl', float('nan')):.1f}")

    sequence = []
    for target in ("Bar_None", "Tempo_120"):
        for name, tid in vocab.items():
            if target in name:
                sequence.append(tid)
                break
    sequence += [groups["pitch"][0], groups["velocity"][0], groups["duration"][0]]

    phase = "pitch"
    n_notes = 0

    for _ in tqdm(range(max_new_tokens), desc="Generating"):
        x = torch.tensor([sequence[-cfg.seq_len:]], dtype=torch.long, device=cfg.device)
        next_logits = gen_model(x)[0, -1]

        allowed = groups["pitch"] + groups["structural"] if phase == "pitch" and n_notes >= 4 else groups[phase]
        next_id = sample_token(next_logits, allowed, temperature, top_k)
        sequence.append(next_id)

        token_name = rev_vocab.get(next_id, "")
        phase = next_phase(token_name)
        if token_name.startswith("Duration_"):
            n_notes += 1

    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    tok([sequence]).dump_midi(output_path)
    print(f"Saved {output_path} ({n_notes} notes)")

In [ ]:
generate(CHECKPOINT_PATH, OUTPUT_PATH, MAX_NEW_TOKENS, TEMPERATURE, TOP_K)